In [102]:
import torch
from torchvision import datasets, transforms
import random
from PIL import Image
import numpy as np
import torch.nn as nn
import torch.nn.functional as F


## Define the transformations to the MINST data

In [103]:
# Define a transformation
transform = transforms.Compose([
    transforms.ToTensor()
])

In [134]:
# Load the MNIST dataset
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

print(train_dataset[0])

(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [ ]:
class Combine(torch.utils.data.Dataset):
    def __init__(self):
        super().__init__()
        self.tf = transforms.ToTensor()
        self.ds = datasets.MNIST(root='.', train=True, transform=self.tf, download=True)

    def __len__(self):
        return len(self.ds)
    

    def __getitem__(self, idx):
        idx = random.sample(range(len(self)), 4)
        store = []
        label = []

        for i in idx:
            x, y = self.ds[i]
            store.append(x)
            label.append(y)

        img = Image.new('L', (56, 56))

        img.paste(store[0], (0, 0))
        img.paste(store[1], (28, 0))
        img.paste(store[2], (0, 28))
        img.paste(store[3], (28, 28))

        return img, label


In [105]:
image_num = 4

random_idx = random.sample(range(len(train_dataset)), image_num)
image_store = []
label_store = []

for idx in random_idx:
    x, y = train_dataset[idx]
    image_store.append(transforms.ToPILImage()(x))
    label_store.append(y)

print(image_store)
print(label_store)

[<PIL.Image.Image image mode=L size=28x28 at 0x13DF57F50>, <PIL.Image.Image image mode=L size=28x28 at 0x13DF56790>, <PIL.Image.Image image mode=L size=28x28 at 0x13DFDBF10>, <PIL.Image.Image image mode=L size=28x28 at 0x14BB24550>]
[3, 4, 3, 6]


In [106]:

img = Image.new('L', (56, 56))
img.paste(image_store[0], (0, 0, 28, 28))      # top-left
img.paste(image_store[1], (28, 0, 56, 28))     # top-right
img.paste(image_store[2], (0, 28, 28, 56))     # bottom-left
img.paste(image_store[3], (28, 28, 56, 56))    # bottom-right

img.show()

In [107]:
grid_size = int(image_num ** 0.5) #16 sqr = 4
patch_size = 28
image_size = grid_size * patch_size

img = Image.new('L', (image_size, image_size))
index = 0
for row in range(grid_size):
    for col in range(grid_size):
        img.paste(image_store[index], (col * patch_size, row * patch_size, (col + 1) * patch_size, (row + 1) * patch_size))
        index += 1

img.show()

In [108]:
smaller_patch_size = 14
overall_grid_size = 16

img = np.array(img).reshape(overall_grid_size, smaller_patch_size, smaller_patch_size)
flattened_patches = torch.tensor(np.array([patch.flatten() for patch in img]), dtype=torch.float32)
print(flattened_patches.shape)

torch.Size([16, 196])


In [109]:

patch_pixel_num = flattened_patches.shape[1]
img_emb_dim = 64
linear_layer = nn.Linear(patch_pixel_num, img_emb_dim)
W_QI = nn.Linear(img_emb_dim, img_emb_dim) 
W_KI = nn.Linear(img_emb_dim, img_emb_dim)
W_VI = nn.Linear(img_emb_dim, img_emb_dim)



In [110]:
img_embeddings = linear_layer(flattened_patches)

print(img_embeddings.shape)
qi = W_QI(img_embeddings)
ki = W_KI(img_embeddings)
vi = W_VI(img_embeddings)


print(q.shape)
print(k.shape)
print(v.shape)

torch.Size([16, 64])
torch.Size([16, 64])
torch.Size([16, 64])
torch.Size([16, 64])


In [127]:
QKI = qi @ ki.T
QKI = QKI / np.sqrt(img_emb_dim)
softmax_QKI = F.softmax(QKI, dim=-1)

img_encoding = softmax_QKI @ vi
print(img_encoding.shape)

torch.Size([16, 64])


In [129]:
img_ff = nn.Sequential(
    nn.Linear(img_emb_dim, img_emb_dim),
    nn.ReLU(),
    nn.Linear(img_emb_dim, img_emb_dim)
)
img_encoding = img_ff(img_encoding)
print(img_encoding.shape)

torch.Size([16, 64])


In [112]:
print(label_store)
id2label = {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: '<s>', 11: '<e>'}
label2id = {v: k for k, v in id2label.items()}

print(id2label)
print(label2id)

[3, 4, 3, 6]
{0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: '<s>', 11: '<e>'}
{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '<s>': 10, '<e>': 11}


In [113]:
label_emb_size = 32
vocab_size = len(id2label)
label_embedding_matrix = nn.Embedding(vocab_size, label_emb_size)


In [114]:
label = [10] + label_store
label = torch.tensor(label)

actual = label_store + [11]
actual = torch.tensor(actual)

print(label)
print(actual)


tensor([10,  3,  4,  3,  6])
tensor([ 3,  4,  3,  6, 11])


In [115]:
label_embedding = label_embedding_matrix(label)
print(label_embedding.shape)


torch.Size([5, 32])


In [132]:
W_QL = nn.Linear(label_emb_size, label_emb_size)
W_KL = nn.Linear(label_emb_size, label_emb_size)
W_VL = nn.Linear(label_emb_size, label_emb_size)


ql = W_QL(label_embedding)
kl = W_KL(label_embedding)
vl = W_VL(label_embedding)

QKL = ql @ kl.T

negative_inf = torch.full_like(QKL, float('-inf'))
masked_QKL = torch.triu(negative_inf, diagonal=1)

QKL = QKL + masked_QKL

QKL = QKL / np.sqrt(label_emb_size)
softmax_QKL = F.softmax(QKL, dim=-1)

label_encoding = softmax_QKL @ vl
print(label_encoding.shape)

torch.Size([5, 32])


## Cross attention block (X)

In [130]:
x_emb_dim = 56
W_QX = nn.Linear(label_emb_size, x_emb_dim)
W_KX = nn.Linear(img_emb_dim, x_emb_dim)
W_VX = nn.Linear(img_emb_dim, label_emb_size)

qx = W_QX(label_encoding)
kx = W_KX(img_encoding)
vx = W_VX(img_encoding)

QKX = qx @ kx.T
QKX = QKX / np.sqrt(x_emb_dim)
softmax_QKX = F.softmax(QKX, dim=-1)

x_encoding = softmax_QKX @ vx
print(x_encoding.shape)

torch.Size([5, 32])


In [131]:
x_ff = nn.Sequential(
    nn.Linear(label_emb_size, label_emb_size),
    nn.ReLU(),
    nn.Linear(label_emb_size, label_emb_size)
)
x_encoding = x_ff(x_encoding)
print(x_encoding.shape)

torch.Size([5, 32])


In [124]:
project_layer = nn.Linear(label_emb_size, vocab_size)

logits = project_layer(x_encoding)
print(logits.shape)


torch.Size([5, 12])


In [126]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, actual)
print(loss)

tensor(7.0230, grad_fn=<NllLossBackward0>)


In [133]:
optimizer = torch.optim.Adam(project_layer.parameters(), lr=1e-3)
optimizer.zero_grad()
loss.backward()
optimizer.step()


In [ ]:
import wandb
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
optim = torch.optim.Adam(
    list(label_embedding_matrix.parameters()) +
    list(linear_layer.parameters()) +
    list(W_QI.parameters()) +
    list(W_KI.parameters()) +
    list(W_VI.parameters()) +
    list(W_QL.parameters()) +
    list(W_KL.parameters()) +
    list(W_VL.parameters()) +
    list(W_QX.parameters()) +
    list(W_KX.parameters()) +
    list(W_VX.parameters()) +
    list(img_ff.parameters()) +
    list(x_ff.parameters()) +
    list(project_layer.parameters()),
    lr=0.001
)
wand

In [ ]:
num_of_epochs = 10

wandb.init(project="mlx5.4-transformers", name=f"gpt_{timestamp}")

for epoch in range(num_of_epochs):
    for 

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{num_of_epochs}, Loss: {loss.item()}")